# 06 — Advise on each element's tolerance

The threshold job. One model call per registered critical data element, advising on the
element's **tolerance** — `config.cde_registry.tolerance_pct`, the share of its rows the
business will accept failing — and writing one row per element to
`results.tolerance_proposal`. A reviewer decides each one on the app's Thresholds page.
**Nothing here changes the register or a rule.**

**Why per element, since 2026-10-08.** Until then this job advised on every rule's own
`fail_threshold_pct`: 86 proposals over 19 elements on its first real pass, 30 asking for
a decision, and eight of the nine that moved a limit simply copied the element's
tolerance onto the rule. The business decides one number per element, and since this
date that number decides what reaches Triage (`v_check_escalation`): a breach whose
element sat within its tolerance is held back, unless the rule is a P1. A rule's limit is
a steward's setting, 0 by default. So the question worth a model call is whether the
tolerance itself is right.

**Three answers, never a higher number.**

* `keep` — the tolerance stands.
* `tighten` — every run sat well inside it, so a regression up to it would go
  unescalated. The proposal is lower.
* `review` — the element sits outside its tolerance (or none is declared), and a person
  must decide whether the data gets fixed or more failure is accepted.

Raising a tolerance is the business accepting worse data; the job may say the question
needs asking, never answer it. `validate` refuses a higher number, and so does a CHECK
constraint on the table.

**The figures come from the views, never from the response.** The current tolerance, the
element's failure rate run by run (`v_element_tolerance`, the scorecard's arithmetic) and
which checks were held back (`v_check_escalation`) are copied into the row from the brief.

**No record value goes to the model.** The brief is the register and run statistics; it
never reads `violation_sample`, and `_no_values` refuses to send anything shaped like an
email, a phone number or a long identifier.

### Before running

Runs as the table's owner today (one-off runs, 2026-10-08). As a scheduled job under a
service principal it needs `SELECT` on the schema and `MODIFY` on
`results.tolerance_proposal` alone — the same shape as the triage job's grant, and like
it not in `07_grants.sql`: giving a job principal a write is a decision.

In [ ]:
from openai import OpenAI
import json, re, uuid, hashlib
import pandas as pd
from datetime import datetime

# TWO LAYOUTS, same as notebook 03. Change SANDPIT, not the constants.
SANDPIT = True

if SANDPIT:
    CATALOG, SCHEMA, PREFIX = "sdpt_data_trnf", "udp_brnz", "dq_"   # rendered by tools/build_production.py
    def _t(group, name): return f"{CATALOG}.{SCHEMA}.{PREFIX}{group}_{name}"
else:
    CATALOG = "dq"
    def _t(group, name): return f"{CATALOG}.{group}.{name}"

CHECK_RUN      = _t("results", "check_run")
ELEMENT_TOL    = _t("results", "v_element_tolerance")
ESCALATION     = _t("results", "v_check_escalation")
PROPOSAL       = _t("results", "tolerance_proposal")
PROPOSAL_CUR   = _t("results", "v_tolerance_proposal_current")
RULE_CURRENT   = _t("config",  "v_rule_registry_current")
CDE_CURRENT    = _t("config",  "v_cde_registry_current")

MODEL_NAME  = "system.ai.gpt-oss-120b"
# gpt-oss reasons before it answers and the reasoning counts against this budget; at
# 1800 the first per-rule run (2026-10-08) lost every large element, two with no answer
# text at all. One element per call is a short answer, so this covers the thinking.
MAX_TOKENS  = 8000
TEMPERATURE = 0.0     # advice that lands in a table a reviewer decides on should be reproducible

NS = uuid.UUID("6f1b4c2e-0000-4000-8000-000000000002")
def det_uuid(*parts): return str(uuid.uuid5(NS, "|".join(parts)))

In [ ]:
def _ctx():
    return dbutils.notebook.entry_point.getDbutils().notebook().getContext()


API_ROOT  = _ctx().apiUrl().get()
API_TOKEN = _ctx().apiToken().get()
client = OpenAI(api_key=API_TOKEN, base_url=f"{API_ROOT}/ai-gateway/mlflow/v1")

# The Lakeflow run id, the portable way: submit with
#     "base_parameters": {"job_run_id": "{{job.run_id}}"}
# currentRunId() is not whitelisted on serverless -- see notebook 03.
dbutils.widgets.text("job_run_id", "")
# Comma-separated cde_ids to advise on, e.g. a rerun of the elements a pass rejected.
# Empty means the whole register.
dbutils.widgets.text("only_cde", "")
ONLY_CDE = {c.strip() for c in dbutils.widgets.get("only_cde").split(",") if c.strip()}
_param = dbutils.widgets.get("job_run_id").strip()
if _param and not _param.startswith("{{"):
    JOB_RUN_ID = _param
else:
    try:
        JOB_RUN_ID = _ctx().currentRunId().toString()
    except Exception as exc:
        JOB_RUN_ID = f"interactive-{uuid.uuid4()}"
        print(f"no job run id ({type(exc).__name__}) - stamping {JOB_RUN_ID}")

PROPOSED_TS = datetime.now().replace(microsecond=0)
PROPOSAL_RUN_ID = det_uuid("tolerance-run", JOB_RUN_ID, PROPOSED_TS.isoformat())
print("Unity Gateway client created ·", MODEL_NAME)
print("job_run_id ·", JOB_RUN_ID, "· proposal_run_id ·", PROPOSAL_RUN_ID)

In [ ]:
# Smoke test -- confirms the endpoint answers before any worklist is built.
print(client.responses.create(
    model=MODEL_NAME, input="Reply with the single word: ready", max_output_tokens=20
).output_text)


---
## 1 · The worklist is the CDE register

One brief per registered element. Its history is `v_element_tolerance` — the element's
failure rate on every run, against the register's current tolerance — and its checks are
the active rules that name it, with what each did on the latest run and whether
`v_check_escalation` held it back.

In [ ]:
elements = spark.sql(f"""
    SELECT cde_id, cde_version, cde_name, data_class, criticality, pii, tolerance_pct,
           definition, regulatory_basis, owner_group
    FROM   {CDE_CURRENT}
    WHERE  status = 'registered'
    ORDER  BY cde_id
""").toPandas()
if ONLY_CDE:
    unknown = ONLY_CDE - set(elements.cde_id)
    if unknown:
        raise ValueError(f"only_cde names elements not registered: {sorted(unknown)}")
    elements = elements[elements.cde_id.isin(ONLY_CDE)].reset_index(drop=True)

history = spark.sql(f"""
    SELECT run_id, run_ts, cde_id, rows_evaluated, rows_failed, score_pct,
           scope_disputed, within_tolerance
    FROM   {ELEMENT_TOL}
""").toPandas()
history["fail_pct"] = 100.0 - history["score_pct"]

latest_run = spark.sql(f"""
    SELECT run_id FROM {CHECK_RUN}
    WHERE  status IN ('pass', 'breach')
    ORDER  BY run_ts DESC LIMIT 1
""").collect()[0]["run_id"]

checks = spark.sql(f"""
    SELECT r.cde_id, r.rule_id, r.rule_name, r.rule_type, r.severity, r.status,
           k.status AS latest_status, k.violation_pct AS latest_violation_pct,
           COALESCE(NOT e.escalates, FALSE) AS held
    FROM   {RULE_CURRENT} r
    LEFT JOIN {CHECK_RUN} k  ON k.rule_id = r.rule_id AND k.run_id = '{latest_run}'
    LEFT JOIN {ESCALATION} e ON e.rule_id = r.rule_id AND e.run_id = '{latest_run}'
    WHERE  r.status = 'active'
    ORDER  BY r.cde_id, r.rule_id
""").toPandas()

print(f"{len(elements)} elements · {len(history)} element-runs · latest run {latest_run} · "
      f"{int(checks.held.sum())} breaching checks held back on it")

---
## 2 · The brief

One page per element: what it is, the tolerance the register declares, where its failure
rate has sat run by run, and its checks on the latest run. Figures only — no rule
expression and no row value.

In [ ]:
def _blank(v, alt=None):
    return alt if v is None or (isinstance(v, float) and v != v) else v


def _r(v, n=4):
    return None if v is None or (isinstance(v, float) and v != v) else round(float(v), n)


def build_payload(e):
    h = history[history.cde_id == e.cde_id].sort_values("run_ts")
    c = checks[checks.cde_id == e.cde_id]
    tol = _blank(e.tolerance_pct)
    stats = None
    if len(h):
        f = h.fail_pct
        stats = {"runs_observed": int(len(h)),
                 "runs_outside": int((~h.within_tolerance.astype(bool)).sum()),
                 "fail_pct_min": _r(f.min()), "fail_pct_median": _r(f.median()),
                 "fail_pct_p90": _r(f.quantile(0.9)), "fail_pct_max": _r(f.max()),
                 "latest_fail_pct": _r(f.iloc[-1]),
                 "last_10": [_r(x, 3) for x in f.tail(10)]}
    return {
        "cde_id": e.cde_id, "cde_version": int(e.cde_version), "cde_name": e.cde_name,
        "data_class": e.data_class, "criticality": e.criticality, "pii": bool(e.pii),
        "tolerance_pct": None if tol is None else float(tol),
        "definition": e.definition, "regulatory_basis": _blank(e.regulatory_basis),
        "owner_group": e.owner_group,
        "scope_disputed": bool(h.scope_disputed.iloc[0]) if len(h) else False,
        "history": stats,
        "checks": [{"rule_id": r.rule_id, "rule_name": r.rule_name, "rule_type": r.rule_type,
                    "severity": r.severity, "latest_status": _blank(r.latest_status),
                    "latest_violation_pct": _r(r.latest_violation_pct),
                    "held": bool(r.held)} for r in c.itertuples()],
    }


def build_brief(p):
    tol = (f"{p['tolerance_pct']}% of the element's rows may fail" if p["tolerance_pct"]
           is not None else "(none declared)")
    h = p["history"]
    hist = ("no check on this element has raised a verdict yet" if h is None else
            f"over {h['runs_observed']} runs the element's failure rate was min "
            f"{h['fail_pct_min']}% / median {h['fail_pct_median']}% / p90 {h['fail_pct_p90']}% "
            f"/ max {h['fail_pct_max']}%, outside the tolerance on {h['runs_outside']}; "
            f"latest {h['latest_fail_pct']}%; last ten runs, oldest first: "
            + ", ".join(f"{x}%" for x in h["last_10"]))
    lines = "".join(
        f"\n  {c['rule_id']} - {c['rule_name']} ({c['rule_type']}, {c['severity']}): "
        + ("did not run" if c["latest_status"] is None else
           f"{c['latest_status']} at {c['latest_violation_pct']}%")
        + (" - HELD BACK from triage, inside the tolerance" if c["held"] else "")
        for c in p["checks"]) or "\n  (no active check)"
    return f"""
Advise on the tolerance of one critical data element.

ELEMENT
-------
{p['cde_id']} - {p['cde_name']}  ({p['data_class']}, criticality {p['criticality']}, PII {p['pii']})
  definition         : {p['definition']}
  regulatory basis   : {p['regulatory_basis'] or '(none recorded)'}
  owning group       : {p['owner_group']}
  declared tolerance : {tol}
  scope dispute      : {'YES - a rule on this element counts rows the register says are legitimately empty, so its failure rate measures the rule, not the data' if p['scope_disputed'] else 'no'}

FAILURE RATE (rows failed over rows checked, across the element's checks, per run)
-------------------------------------------------------------------------------------
  {hist}

CHECKS ON THE LATEST RUN
------------------------{lines}

TASK
----
Advise keep, tighten or review on the declared tolerance, with the figures that decide
it. Return only the requested JSON object.
"""


payloads = [build_payload(e) for e in elements.itertuples()]
print(f"{len(payloads)} briefs assembled")
if payloads:
    print(build_brief(payloads[0]))

In [ ]:
SYSTEM_PROMPT = '''
You are an experienced enterprise Data Quality Analyst advising a business on the
TOLERANCE of one critical data element: the share of its rows the business accepts
failing. The tolerance decides which failing checks reach the people who fix data -
a failing check whose element sits within it is held back, unless the check is P1.

1.  Base your advice only on the evidence supplied. Do not invent facts, and do not
    describe values you were not given.
2.  Answer with exactly one of:
    keep    - the tolerance stands. proposed_tolerance_pct equals the current one.
    tighten - the failure rate has sat well inside the tolerance on every run, so a
              regression up to it would go unescalated. Propose a LOWER figure that is
              still above every run in the history, so nothing escalates today that
              did not before. proposed_tolerance_pct is lower than the current one.
    review  - the element sits outside its tolerance on most runs or the latest, or no
              tolerance is declared. A person must decide whether the data gets fixed
              or the business accepts more failure. proposed_tolerance_pct equals the
              current one (null if none is declared).
3.  NEVER propose a higher tolerance. Raising it is the business accepting worse data.
    You may say the question needs asking (review); you may not answer it.
4.  If there is a scope dispute, the failure rate measures a rule defect, not the data.
    Answer keep, and say the rule's scope must be fixed before the tolerance can be
    judged. Never tighten on a disputed element.
5.  A tolerance of 0 cannot be tightened. An element whose failures came and went and
    whose latest run is inside the tolerance is usually keep: a fixed defect is not a
    reason to move the number.
6.  Where no check has raised a verdict, answer keep: there is no basis yet.
7.  rationale quotes the figures it rests on, in two to four sentences a business
    reader can follow. reviewer is the team that should decide - the element's owning
    group unless there is a reason otherwise.
8.  You advise. Nothing you return changes the register; a person decides, and a second
    person approves any change to it.
9.  Return only valid JSON, exactly this structure:

{
  "advice": "keep | tighten | review",
  "proposed_tolerance_pct": 0.0,
  "rationale": "the figures this rests on",
  "reviewer": "team to decide it",
  "reasoning": "brief explanation linking evidence to the advice"
}
'''

---
## 3 · Call, validate, and drop what does not validate

`validate` is the gate, and it repeats the table's CHECK constraints so a bad answer is
dropped here, with its reason printed, rather than refused at the INSERT after every call
has been paid for. The one that matters: never a higher number.

In [ ]:
ADVICE = ["keep", "tighten", "review"]


def parse_llm_json(text):
    text = re.sub(r"\s*```$", "", re.sub(r"^```(json)?\s*", "", text.strip()))
    return json.loads(text)


def validate(rec, payload):
    problems = []
    cur = payload["tolerance_pct"]
    advice = rec.get("advice")
    if advice not in ADVICE:
        return [f"advice {advice!r} not in {ADVICE}"]
    raw = rec.get("proposed_tolerance_pct")
    try:
        new = None if raw is None else float(raw)
    except (TypeError, ValueError):
        return [f"proposed_tolerance_pct {raw!r} not numeric"]
    if new is not None and not 0.0 <= new <= 100.0:
        problems.append(f"{new} out of range")
    if cur is not None and (new is None or new > cur):
        problems.append(f"{new}% raises or drops the {cur}% tolerance -- never a higher number")
    if advice == "tighten":
        if cur is None or new is None or not new < cur:
            problems.append(f"tighten must lower the tolerance ({cur} -> {new})")
        if payload["scope_disputed"]:
            problems.append("tighten on a scope-disputed element (prompt rule 4)")
    elif new != cur:
        problems.append(f"{advice} must leave the number alone ({cur} -> {new})")
    for f in ("rationale", "reviewer", "reasoning"):
        if not str(rec.get(f) or "").strip():
            problems.append(f"no {f}")
    return problems


# No customer value goes to the model. The brief is built from the registers and run
# statistics only, and never reads violation_sample -- but an element definition typed
# into the register could still carry one. So the text is checked on its way out:
# anything shaped like an email address, an Australian phone number or a long
# identifier stops the call.
_VALUE_SHAPES = {
    "email address": re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    "phone number":  re.compile(r"(?<![\w])(?:\+?61|0)[2-478]\d{8}(?![\w])"),
    "long identifier": re.compile(r"(?<![\w.%{])\d{9,}(?![\w])"),
}


def _no_values(brief):
    # The refusal names the kind of value, never the value: it lands in the job log.
    found = [k for k, rx in _VALUE_SHAPES.items() if rx.search(brief)]
    if found:
        raise ValueError("brief not sent: it carries what looks like a record value ("
                         + ", ".join(found) + ")")


def _ask(brief):
    _no_values(brief)
    resp = client.responses.create(
        model=MODEL_NAME, instructions=SYSTEM_PROMPT, input=brief,
        temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS,
    )
    # A cut-off answer is reported as one, not as the JSON error it would otherwise raise.
    if getattr(resp, "status", None) == "incomplete":
        why = getattr(getattr(resp, "incomplete_details", None), "reason", None)
        raise ValueError(f"response incomplete ({why}) at max_output_tokens={MAX_TOKENS}")
    return resp.output_text


advised, rejected = [], []
for i, p in enumerate(payloads, 1):
    print(f"[{i}/{len(payloads)}] {p['cde_id']}", end=" ")
    try:
        brief = build_brief(p)
        rec = parse_llm_json(_ask(brief))
        problems = validate(rec, p)
        if problems:
            raise ValueError("; ".join(problems))
        advised.append({"payload": p, "rec": rec, "brief": brief})
        print(f"-> {rec['advice']}"
              + (f" {p['tolerance_pct']}% -> {rec['proposed_tolerance_pct']}%"
                 if rec["advice"] == "tighten" else ""))
    except Exception as e:
        rejected.append({"cde_id": p["cde_id"], "error": str(e)})
        print(f"-> REJECTED: {e}")

print(f"\n{len(advised)} elements advised, {len(rejected)} rejected")

In [ ]:
# Rejections FIRST and unconditionally -- the run where nothing validated is the run
# whose reason you most need (notebook 03 learned this the expensive way).
if rejected:
    print(f"{len(rejected)} element(s) rejected by validate() -- these wrote nothing:")
    for r in rejected:
        print(f"  {r['cde_id']}  {r['error']}")

if not advised:
    print("\nNothing was advised. Every rejection is listed above; each is a defect in "
          "the response or in the validator, not a reason to relax the gate.")
else:
    for a in advised:
        p, rec = a["payload"], a["rec"]
        print(f"  {p['cde_id']:24} {rec['advice']:8} {p['tolerance_pct']} -> "
              f"{rec['proposed_tolerance_pct']}  ({rec['reviewer']})")

---
## 4 · Write

`INSERT`, never `MERGE`: a proposal is advice at a moment, and the reviewer's decision
references it by id. `proposal_id` is deterministic on the proposal run and the element,
so re-running a failed write cannot double a proposal. The current tolerance and every
statistic come from the brief, never from the response.

In [ ]:
rows = []
for a in advised:
    p, rec = a["payload"], a["rec"]
    h = p["history"] or {}
    new = rec["proposed_tolerance_pct"]
    rows.append({
        "proposal_id": det_uuid("tolerance", PROPOSAL_RUN_ID, p["cde_id"]),
        "proposal_run_id": PROPOSAL_RUN_ID,
        "proposed_ts": PROPOSED_TS,
        "cde_id": p["cde_id"],
        "cde_version": p["cde_version"],
        "current_tolerance_pct": p["tolerance_pct"],
        "proposed_tolerance_pct": None if new is None else float(new),
        "advice": rec["advice"],
        "rationale": str(rec["rationale"]),
        "reviewer": str(rec["reviewer"]),
        "runs_observed": h.get("runs_observed"),
        "runs_outside": h.get("runs_outside"),
        "fail_pct_min": h.get("fail_pct_min"),
        "fail_pct_median": h.get("fail_pct_median"),
        "fail_pct_p90": h.get("fail_pct_p90"),
        "fail_pct_max": h.get("fail_pct_max"),
        "latest_fail_pct": h.get("latest_fail_pct"),
        "scope_disputed": p["scope_disputed"],
        "model_endpoint": MODEL_NAME,
        "model_input_payload": json.dumps({
            "reasoning": rec["reasoning"],
            "system_prompt_sha": hashlib.sha256(SYSTEM_PROMPT.encode()).hexdigest()[:16],
            "model": MODEL_NAME, "temperature": TEMPERATURE, "brief": a["brief"],
        }),
        "job_run_id": JOB_RUN_ID,
    })

if rows:
    # Schema from the table, not inferred -- a nullable column that is None in every row
    # fails inference after every model call has been paid for.
    candidates = spark.createDataFrame(rows, schema=spark.table(PROPOSAL).schema)
    candidates.createOrReplaceTempView("candidate_tolerance")
    print(f"{len(rows)} proposals to write")
    spark.sql(f"""
    INSERT INTO {PROPOSAL}
    SELECT c.* FROM candidate_tolerance c
    WHERE NOT EXISTS (SELECT 1 FROM {PROPOSAL} e WHERE e.proposal_id = c.proposal_id)
    """)
print(spark.sql(f"SELECT COUNT(*) AS n FROM {PROPOSAL} WHERE proposal_run_id = '{PROPOSAL_RUN_ID}'")
      .collect()[0]["n"], "proposals on this run")

In [ ]:
# Verification: what the app's Thresholds page will show -- one row per element.
display(spark.sql(f"""
    SELECT cde_id, current_tolerance_pct, proposed_tolerance_pct, advice, review_state,
           runs_outside, runs_observed, latest_fail_pct, reviewer
    FROM   {PROPOSAL_CUR}
    WHERE  proposal_run_id = '{PROPOSAL_RUN_ID}'
    ORDER  BY review_state, cde_id
"""))